In [2]:
# Cell 0 — Setup
from pathlib import Path
import sys, json
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")


True

In [3]:

# Cell 1 — Build pipeline + evaluator
from src.retrieval import HybridRetriever
from src.generation import Generator
from src.pipeline import RAGPipeline
from src.evaluation import (
    TestQuestion, LLMJudge, Evaluator, summarize_results, save_results
)

retriever = HybridRetriever(
    chroma_path=str(PROJECT_ROOT / "chroma_db"),
    collection_name="financial_risk_10k",
)
generator = Generator(model_name="gemini-2.5-flash-lite")
pipeline = RAGPipeline(retriever=retriever, generator=generator, top_k=5)
judge = LLMJudge(model_name="gemini-2.5-flash")  # Bigger model for judging

evaluator = Evaluator(pipeline=pipeline, judge=judge, delay_seconds=5)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading chunks from collection 'financial_risk_10k'...
Building BM25 index over 1401 chunks...
✓ HybridRetriever ready: 1401 chunks


In [ ]:

# Cell 2 — Load test set
test_data = json.loads((PROJECT_ROOT / "data" / "test_set"/"evaluation_test_set.json").read_text())
questions = [TestQuestion(**row) for row in test_data]
print(f"Loaded {len(questions)} test questions")


Loaded 18 test questions


In [5]:

# Cell 3 — Run evaluation
# At ~15s per question (with rate limiting), 25 questions ≈ 6-8 minutes
results = evaluator.evaluate_all(questions)


[1/18] single_01: What cybersecurity risks does JPMorgan disclose in its Risk ...
  ❌ Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
[2/18] single_02: How does Capital One describe its commercial real estate exp...
  ❌ Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
[3/18] single_03: What climate-related risks does BlackRock identify?...
  P@5=1.00  R@5=1.00  faith=5  rel=5
[4/18] single_04: What are PayPal's main risks from third-party service provid...
  ❌ Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
[5/18] single_05: What int

KeyboardInterrupt: 

In [ ]:

# Cell 4 — Summary
summary = summarize_results(results)
print(json.dumps(summary, indent=2))


{
  "n_questions": 4,
  "retrieval": {
    "avg_precision_at_5": 0.45,
    "avg_recall_at_5": 0.75,
    "avg_mrr": 0.5
  },
  "generation": {
    "avg_faithfulness": 5.0,
    "avg_relevance": 2.3333333333333335,
    "citation_hallucinations": 0,
    "refusal_accuracy": 0.75
  },
  "avg_latency_ms": 1346.0058569908142
}


In [6]:

# Cell 5 — Save for future comparison
save_results(results, str(PROJECT_ROOT / "data" / "eval_run_baseline.json"))


✓ Saved 4 results to e:\P1_JOB\AI projects\financial_risk_rag\data\eval_run_baseline.json


In [7]:

# Cell 6 — Inspect failures
print("\n=== FAILURES TO INVESTIGATE ===")
for r in results:
    fails = []
    if r.precision_at_5 < 0.5:
        fails.append(f"P@5={r.precision_at_5:.2f}")
    if r.faithfulness_score and r.faithfulness_score <= 3:
        fails.append(f"faith={r.faithfulness_score}")
    if r.citation_hallucination:
        fails.append("cite_halluc")
    if r.refusal_correct is False:
        fails.append("wrong_refusal")
    if fails:
        print(f"\n{r.question_id} [{r.category}]: {', '.join(fails)}")
        print(f"  Q: {r.question}")
        print(f"  Retrieved: {r.retrieved_tickers}")
        print(f"  Answer: {r.answer[:200]}...")


=== FAILURES TO INVESTIGATE ===

specific_01 [specific_term]: P@5=0.00, wrong_refusal
  Q: Which filings mention LIBOR or SOFR transition?
  Retrieved: ['BLK', 'COF', 'COF', 'COF', 'COF']
  Answer: The provided excerpts do not contain sufficient information to answer this question....

out_of_scope_01 [out_of_scope]: P@5=0.00
  Q: What was PayPal's exact net income in Q3 2024?
  Retrieved: ['PYPL', 'PYPL', 'PYPL', 'BLK', 'PYPL']
  Answer: The provided excerpts do not contain sufficient information to answer this question....


In [ ]:
# Search for LIBOR or SOFR in chunks
hits = [
    (m.get("ticker"), text[:200])
    for m, text in zip(retriever.chunk_metas, retriever.chunk_texts)
    if "LIBOR" in text or "SOFR" in text
]
print(f"Chunks mentioning LIBOR/SOFR: {len(hits)}")
for ticker, snippet in hits[:5]:
    print(f"\n[{ticker}] {snippet}...")

Chunks mentioning LIBOR/SOFR: 0


: 